In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "dataset").exists() and (p / "figures").exists():
            return p

    for p in [cwd, *cwd.parents]:
        candidate = p / "analysis" / "notebooks"
        if candidate.exists():
            return candidate

    return cwd

NOTEBOOKS_DIR = find_notebooks_dir()
DATASET_PATH = NOTEBOOKS_DIR / "dataset" / "yandex_music_data.json"
FIG_DIR = NOTEBOOKS_DIR / "figures"
EXPORTS_DIR = NOTEBOOKS_DIR / "exports"

FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("DATASET_PATH:", DATASET_PATH)
print("FIG_DIR:", FIG_DIR)

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

tracks = pd.DataFrame(data.get("tracks", []))
likes = pd.DataFrame(data.get("likes", []))

likes["liked_at"] = pd.to_datetime(likes["liked_at"], utc=True, errors="coerce")
tracks["id"] = tracks["id"].astype("int64")
likes["track_id"] = likes["track_id"].astype("int64")

df = likes.merge(tracks, left_on="track_id", right_on="id", how="left")

df["primary_genre"] = df["primary_genre"].fillna(
    df["genres"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
)

def first_artist(artists):
    if isinstance(artists, list) and len(artists) > 0:
        return artists[0].get("name")
    return None

df["main_artist"] = df["artists"].apply(first_artist)

def savefig(name: str):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    return out

print("tracks:", len(tracks), "likes:", len(likes), "merged:", len(df))
df.head()


NOTEBOOKS_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks
DATASET_PATH: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\dataset\yandex_music_data.json
FIG_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\figures
tracks: 1026 likes: 1026 merged: 1026


,track_id,liked_at,id,title,duration_ms,explicit,primary_genre,release_year,genres,artists,albums,main_artist
0,143449115,2025-10-03 15:02:01+00:00,143449115,MARTINE ROSE,186890,True,rusrap,2025.0,[rusrap],"[{'id': 13992820, 'name': 'madk1d'}, {'id': 82...","[{'id': 38435712, 'title': 'MARTINE ROSE', 'ge...",madk1d
1,330817,2025-10-03 09:52:49+00:00,330817,Let Down,299260,False,indie,1997.0,[indie],"[{'id': 36825, 'name': 'Radiohead'}]","[{'id': 3389007, 'title': 'OK Computer', 'genr...",Radiohead
2,10169820,2025-09-29 21:03:59+00:00,10169820,Alive,204540,False,electronics,2013.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 1182273, 'title': 'Ice On The Dune', '...",Empire Of The Sun
3,332895,2025-09-29 14:47:22+00:00,332895,We Are The People,267360,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun
4,332764,2025-09-28 13:22:49+00:00,332764,Walking On A Dream,196340,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun


In [2]:
liked_ids = set(df["track_id"].dropna().astype("int64").tolist())

top_genres = df["primary_genre"].value_counts().head(10)
user_top5_genres = top_genres.head(5).index.tolist()

print("User liked tracks:", len(liked_ids))
print("Top-5 genres:", user_top5_genres)

display(pd.DataFrame({
    "top_genre": top_genres.index,
    "likes": top_genres.values
}).head(10))


User liked tracks: 1026
Top-5 genres: ['indie', 'rusrap', 'rock', 'pop', 'alternative']


,top_genre,likes
0,indie,154
1,rusrap,138
2,rock,99
3,pop,98
4,alternative,64
5,allrock,51
6,rap,49
7,rusrock,45
8,electronics,32
9,ruspop,32


In [3]:
import random

rng = random.Random(99)

genres_pool = [g for g in df["primary_genre"].dropna().unique().tolist() if isinstance(g, str)]
if len(genres_pool) < 10:
    genres_pool = ["indie","rock","pop","rap","electronics","soundtrack","alternative","jazz","country","dance"]

all_track_ids = tracks["id"].dropna().astype("int64").tolist()
max_real_id = int(max(all_track_ids)) if len(all_track_ids) else 1_000_000
fake_catalog_ids = list(range(max_real_id + 20_000, max_real_id + 20_600))

def sample_genre_alice(genre_focus: float = 0.45) -> str:
    """
    У Алисы ниже "прилипание" к top-5 жанрам -> больше разнообразия.
    """
    if user_top5_genres and rng.random() < genre_focus:
        return rng.choice(user_top5_genres)
    return rng.choice(genres_pool)

def make_alice_recommendations(n: int = 80, overlap_with_likes: float = 0.20, genre_focus: float = 0.45):
    n_overlap = int(n * overlap_with_likes)

    overlap_ids = rng.sample(list(liked_ids), k=min(n_overlap, len(liked_ids)))
    new_ids = rng.sample(fake_catalog_ids, k=n - len(overlap_ids))

    recos = []

    for tid in overlap_ids:
        row = tracks.loc[tracks["id"] == tid]
        genre = None
        if not row.empty:
            genre = row.iloc[0].get("primary_genre")

        recos.append({
            "algo": "Алиса",
            "track_id": int(tid),
            "is_from_likes": True,
            "genre": genre if genre else sample_genre_alice(genre_focus),
            "score": rng.random() * 0.25 + 0.65  # чуть ниже, чем у "волны"
        })

    for tid in new_ids:
        recos.append({
            "algo": "Алиса",
            "track_id": int(tid),
            "is_from_likes": False,
            "genre": sample_genre_alice(genre_focus),
            "score": rng.random() * 0.85 + 0.10
        })

    recos.sort(key=lambda x: x["score"], reverse=True)
    return pd.DataFrame(recos)

reco_alice = make_alice_recommendations(n=80, overlap_with_likes=0.20, genre_focus=0.45)

reco_alice.head(10)


,algo,track_id,is_from_likes,genre,score
0,Алиса,143469279,False,indie,0.949608
1,Алиса,143469646,False,rock,0.907417
2,Алиса,143469630,False,folkrock,0.906066
3,Алиса,256700,True,rock,0.897780
4,Алиса,131437513,True,relax,0.881033
5,Алиса,19794009,True,local-indie,0.870108
6,Алиса,27797461,True,rock,0.864467
7,Алиса,143469138,False,films,0.837736
8,Алиса,143469390,False,alternative,0.833301
9,Алиса,21739172,True,electronics,0.819581


In [4]:
def shannon_entropy(counts) -> float:
    p = np.array(counts, dtype=float)
    p = p[p > 0]
    if p.sum() == 0:
        return 0.0
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum())

def alice_metrics(reco_df: pd.DataFrame) -> dict:
    reco_set = set(reco_df["track_id"].tolist())
    overlap = len(reco_set & liked_ids)
    novelty = 1 - overlap / len(reco_set) if len(reco_set) else 0.0

    genre_counts = reco_df["genre"].value_counts()
    entropy = shannon_entropy(genre_counts.values)

    in_top5 = reco_df["genre"].isin(user_top5_genres).mean() if user_top5_genres else 0.0

    return {
        "n_reco": int(len(reco_set)),
        "overlap_with_likes": int(overlap),
        "novelty_rate": float(novelty),
        "top5_genre_match_rate": float(in_top5),
        "n_genres": int(genre_counts.shape[0]),
        "genre_entropy": float(entropy),
        "avg_score": float(reco_df["score"].mean()),
        "median_score": float(reco_df["score"].median()),
    }

metrics = alice_metrics(reco_alice)
pd.DataFrame([metrics])


,n_reco,overlap_with_likes,novelty_rate,top5_genre_match_rate,n_genres,genre_entropy,avg_score,median_score
0,80,16,0.8,0.3875,35,4.716648,0.542307,0.581255


In [5]:
genre_top = reco_alice["genre"].value_counts().head(12).sort_values()

plt.figure(figsize=(10,6))
plt.barh(genre_top.index, genre_top.values)
plt.title("Алиса: топ жанров в рекомендациях")
plt.xlabel("Количество треков")
savefig("alice_genres.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/alice_genres.png')

In [6]:
cnt_repeat = int(reco_alice["is_from_likes"].sum())
cnt_new = int((~reco_alice["is_from_likes"]).sum())

plt.figure(figsize=(6,6))
plt.pie([cnt_new, cnt_repeat], labels=["Новые треки", "Уже в лайках"], autopct="%1.1f%%")
plt.title("Алиса: доля новых треков (novelty)")
savefig("alice_novelty.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/alice_novelty.png')

In [7]:
plt.figure(figsize=(10,4))
plt.hist(reco_alice["score"], bins=20)
plt.title("Алиса: распределение score в рекомендациях (демо)")
plt.xlabel("Score")
plt.ylabel("Количество треков")
savefig("alice_score_hist.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/alice_score_hist.png')

In [8]:
in_top5 = reco_alice["genre"].isin(user_top5_genres).value_counts()

labels = ["Другие жанры", "Top-5 жанров пользователя"]
values = [int(in_top5.get(False, 0)), int(in_top5.get(True, 0))]

plt.figure(figsize=(7,4))
plt.bar(labels, values)
plt.title("Алиса: совпадение с любимыми жанрами")
plt.ylabel("Количество треков")
savefig("alice_top5_genres_match.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/alice_top5_genres_match.png')

In [9]:
out_csv = EXPORTS_DIR / "reco_alice_demo.csv"
reco_alice.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)


Saved: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\exports\reco_alice_demo.csv


In [10]:
print(
    f"Алиса (демо): {metrics['n_reco']} рекомендаций, "
    f"пересечение с лайками = {metrics['overlap_with_likes']} "
    f"({(1-metrics['novelty_rate'])*100:.1f}% повторов), "
    f"novelty = {metrics['novelty_rate']*100:.1f}%, "
    f"попадание в топ-5 жанров = {metrics['top5_genre_match_rate']*100:.1f}%, "
    f"разнообразие жанров: {metrics['n_genres']} жанров, энтропия = {metrics['genre_entropy']:.2f}."
)


Алиса (демо): 80 рекомендаций, пересечение с лайками = 16 (20.0% повторов), novelty = 80.0%, попадание в топ-5 жанров = 38.8%, разнообразие жанров: 35 жанров, энтропия = 4.72.
